# EDA: Provider Log

## Setup
Reset database and seed base data.

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 3d254eb0-d290-4cb1-aa19-c7a5160d357a
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 23bcf70a-474e-4104-b5ea-32aa38331970
Seeded SystemPrompt 'format' with ID: 1 and GUID: c4b89306-990a-45e4-8164-e8fac6b4afa7
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 43d0817e-5ad4-4e95-aaae-775ee2e6e423
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/session_linked_execution.py")
file.write_text("def ping( user ):\n return f\"pong {user}\"")

# Construct input using session metadata
input_data = {
    "file_name":  str(file),
    "session_id": session_row.id,
    "system":     session_row.name
}

# Run associated program
program = ProgramProviderFactory.create(id=session_row.program_provider_id)
result = program.run(input_data)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


📜 No conversation log entry block found in response.


✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "reason": "success",
  "decision": "final",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_name": "tests\\session_linked_execution.py",
    "working_file": null,
    "session_id": 1,
    "system": "codecritic_test_session",
    "reason": "preprocessing complete",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing",
    "output": {}
  },
  "provider_name": "codecritic_program"
}


### Load raw log entries

In [3]:
import sqlite3
import pandas as pd
import json
from app.db.connection import DB_PATH

# Connect and query
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM provider_log WHERE session_id='1'", conn)
conn.close()

# Pretty-print each log entry
for i, row in df.iterrows():
    print(f"\n🔹 Entry {i + 1} — ID {row['id']}")
    print(f"🕒 Timestamp: {row['timestamp']}")
    print(f"🧩 Provider ID: {row['provider_id']} ({row['provider_type']})")
    print(f"📦 Output Schema: {row['output_schema']}")
    print(f"⏱️ Latency: {row['latency_ms']} ms")
    print(f"📄 File Name: {row['file_name']}")
    print(f"🔑 Config Hash: {row['config_hash']}")
    
    # 🔁 Caller trace
    called_by_type = row.get("called_by_type", None)
    called_by_id = row.get("called_by_id", None)
    if called_by_type or called_by_id:
        print(f"🧭 Called By: {called_by_type} (ID {called_by_id})")

    # Safely parse JSON input/output if possible
    try:
        parsed_input = json.loads(row['input'])
        parsed_output = json.loads(row['output'])
        print("📥 Input:")
        print(json.dumps(parsed_input, indent=2))
        print("📤 Output:")
        print(json.dumps(parsed_output, indent=2))
    except Exception:
        print(f"📥 Input (raw): {row['input']}")
        print(f"📤 Output (raw): {row['output']}")

    print("—" * 80)



🔹 Entry 1 — ID 1
🕒 Timestamp: 2025-06-03T13:39:07.800190+00:00
🧩 Provider ID: 1 (PROVIDER_TYPE.TOOL)
📦 Output Schema: ToolOutputSchema
⏱️ Latency: 229 ms
📄 File Name: de021b43-40c2-47a1-b830-118a7ec32eeb.py
🔑 Config Hash: None
🧭 Called By: PROVIDER_TYPE.SCORE (ID 2)
📥 Input:
{
  "target": "C:\\Repos\\codecritic\\tests\\session_linked_execution.py",
  "check": true
}
📤 Output:
{
  "return_code": 0,
  "stdout": "--- C:\\Repos\\codecritic\\tests\\session_linked_execution.py\t2025-06-03 13:39:07.134638+00:00\n+++ C:\\Repos\\codecritic\\tests\\session_linked_execution.py\t2025-06-03 13:39:08.006101+00:00\n@@ -1,2 +1,2 @@\n-def ping( user ):\n- return f\"pong {user}\"\n\\ No newline at end of file\n+def ping(user):\n+    return f\"pong {user}\"",
  "stderr": "Identified `C:\\Repos\\codecritic` as project root containing a .git directory.\nFound input source: \"C:\\Repos\\codecritic\\tests\\session_linked_execution.py\"\nwould reformat C:\\Repos\\codecritic\\tests\\session_linked_execution.p

### Parse enum fields

In [4]:
from app.enums import logging_enums, fsm_enums, agent_enums
print('Columns:', df.columns.tolist())

Columns: ['id', 'session_id', 'timestamp', 'provider_id', 'provider_type', 'input', 'output', 'output_schema', 'latency_ms', 'config_hash', 'file_name', 'called_by_type', 'called_by_id']


### Validate field values

In [5]:
print(df.isnull().sum())

id                0
session_id        0
timestamp         0
provider_id       0
provider_type     0
input             0
output            0
output_schema     0
latency_ms        0
config_hash       3
file_name         0
called_by_type    0
called_by_id      0
dtype: int64


### Basic counts

In [6]:
print(df.shape)
print(df['timestamp'].min(), df['timestamp'].max())

(8, 13)
2025-06-03T13:39:07.782461+00:00 2025-06-03T13:39:08.040292+00:00
